In [1]:
import geopandas as gpd
import pandas as pd

from utils.disparity_ratio import compute_weighted_exposure_stats

In [2]:
# load data layers
datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
study_period = '2008_2012'  #'2013_2017' #'2018_2022' ##  # #
outfilepth = f'{datadir}rsei_weighted_disparity_ratios_huc12_{study_period}.csv'
metric = f'{study_period}_TOXCONC'
# load and merge the data
gdf = gpd.read_file(datadir + 'aoi_huc12_boundaries.gpkg')
huc12_rsei = pd.read_csv(
    datadir + 'huc12_rsei_toxconc_weighted.csv', dtype={'huc12': str}, index_col=0)
huc12_demographics = pd.read_csv(
    datadir + 'huc12_demographics.csv', dtype={'huc12': str}, index_col=0)
# combine tabular and spatial data using huc12 ids
gdf = gdf.merge(huc12_rsei, on='huc12',  suffixes=('', '_DROP'))
gdf = gdf.merge(huc12_demographics, on='huc12', suffixes=('', '_DROP'))
gdf = gdf.filter(regex='^(?!.*_DROP)')

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


In [3]:
# initiate a list to append all results to
RESULTS = []

### using all HUC12 units

In [4]:
n = len(gdf)
population_examined = gdf[f'{study_period}_total'].sum()
analysis = f'using all HUC12 units, N = {n}, population = {population_examined}'
print (analysis)
result = compute_weighted_exposure_stats(gdf, study_period=study_period, metric=metric)
result['analysis'] = analysis
RESULTS.append(result)

using all HUC12 units, N = 2576, population = 11956439.521913934


### using HUC12 units with non zero TOXCONC

In [5]:
gdf_selected = gdf[~gdf[metric].isnull()].copy()
n = len(gdf_selected)
population_examined = gdf_selected[f'{study_period}_total'].sum()
analysis = f'using HUC12 units with non zero TOXCONC, N = {n}, population = {population_examined}'
print (analysis)
result = compute_weighted_exposure_stats(gdf_selected, study_period=study_period, metric=metric)
result['analysis'] = analysis
RESULTS.append(result)

using HUC12 units with non zero TOXCONC, N = 745, population = 6114176.606231747


### using 100 HUC12 units with the highest TOXCONC

In [6]:
gdf_selected = gdf.sort_values(by=metric, ascending=False).head(100)
n = len(gdf_selected)
population_examined = gdf_selected[f'{study_period}_total'].sum()
analysis = f'using 100 HUC12 units with the highest TOXCONC, N = {n}, population = {population_examined}'
print (analysis)
result = compute_weighted_exposure_stats(gdf_selected, study_period=study_period, metric=metric)
result['analysis'] = analysis
RESULTS.append(result)

using 100 HUC12 units with the highest TOXCONC, N = 100, population = 1624099.552843362


### using 200 HUC12 units with the highest TOXCONC

In [7]:
gdf_selected = gdf.sort_values(by=metric, ascending=False).head(200)
n = len(gdf_selected)
population_examined = gdf_selected[f'{study_period}_total'].sum()
analysis = f'using 200 HUC12 units with the highest TOXCONC, N = {n}, population = {population_examined}'
print (analysis)
result = compute_weighted_exposure_stats(gdf_selected, study_period=study_period, metric=metric)
result['analysis'] = analysis
RESULTS.append(result)

using 200 HUC12 units with the highest TOXCONC, N = 200, population = 3010646.041553813


### using HUC12 units within the 90th percentile TOXCONC

In [8]:
# Calculate the 90th percentile of the 'risk' column
cutoff = gdf[metric].quantile(0.90)
# Create a new DataFrame with rows where 'risk' is greater than or equal to the 90th percentile
gdf_selected = gdf[gdf[metric] >= cutoff]
n = len(gdf_selected)
population_examined = gdf_selected[f'{study_period}_total'].sum()
analysis = f'using HUC12 units within the 90th percentile TOXCONC, N = {n}, population = {population_examined}'
print (analysis)
result = compute_weighted_exposure_stats(gdf_selected, study_period=study_period, metric=metric)
result['analysis'] = analysis
RESULTS.append(result)

using HUC12 units within the 90th percentile TOXCONC, N = 75, population = 1477429.1046913653


### compute by Qualitative Region

In [9]:
regions = [
    'Headwaters', 'Gorge', 'Driftless', 'Working River', 'Confluence',
    'Chickasaw', 'Delta', 'Lower Mississippi', 'Gulf South'
]
for region in regions:
    gdf_selected = gdf[gdf['Region']==region]
    n = len(gdf_selected)
    population_examined = gdf_selected[f'{study_period}_total'].sum()
    analysis = f'Regional: {region}, N = {n}, population = {population_examined}'
    print (analysis)
    result = compute_weighted_exposure_stats(gdf_selected, study_period=study_period, metric=metric)
    result['analysis'] = analysis
    RESULTS.append(result)

Regional: Headwaters, N = 314, population = 208355.32405301018
Regional: Gorge, N = 240, population = 3359792.72364721
Regional: Driftless, N = 430, population = 708739.9693992231
Regional: Working River, N = 390, population = 852251.6119891694
Regional: Confluence, N = 344, population = 2979302.316764746
Regional: Chickasaw, N = 111, population = 272693.9124743761
Regional: Delta, N = 365, population = 1559112.8943245783
Regional: Lower Mississippi, N = 189, population = 221451.53222454304
Regional: Gulf South, N = 193, population = 1794739.2370370787


In [10]:
pd.concat(RESULTS).reset_index().to_csv(outfilepth)